# 🌦️ Dashboard Climático Interactivo: Análisis de Cúcuta
---
**Propósito:** Este proyecto analiza datos climáticos históricos y de pronóstico utilizando la API de Open-Meteo. El objetivo es transformar datos crudos en un dashboard interactivo que permita identificar patrones de temperatura, humedad y precipitación.

<div style="background-color: #e6f2ff; padding: 10px; border-radius: 5px; border-left: 5px solid #1e90ff; margin: 10px 0; color: #000000;">
    <strong>📅 Fechas importantes:</strong> 06/05/2026 y 13/05/2026
</div>

**Tecnologías usadas:** Python, Pandas, Plotly (Interatividad), Open-Meteo API.

In [51]:
import openmeteo_requests
import pandas as pd
import requests_cache
import os
from retry_requests import retry
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurar cliente con caché para no saturar la API
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

print("✅ Librerías cargadas y cliente API configurado.")
# Creamos carpetas para organizar el proyecto
# os.makedirs("../data", exist_ok=True)

✅ Librerías cargadas y cliente API configurado.


In [52]:
# 📌 Configuración de la API
"""
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 7.9,
    "longitude": -72.5,
    "hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "rain"],
    "timezone": "America/Bogota",
    #"past_days": 7, # Aumentamos a 7 días para mejor análisis
    #"forecast_days": 0
    "start_date": "2026-05-06", 
    "end_date": "2026-05-13"
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

# Procesar variables horarias
hourly = response.Hourly()
hourly_data = {
    "datetime": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    ),
    "temperature": hourly.Variables(0).ValuesAsNumpy(),
    "humidity": hourly.Variables(1).ValuesAsNumpy(),
    "precipitation": hourly.Variables(2).ValuesAsNumpy(),
    "rain": hourly.Variables(3).ValuesAsNumpy()
}

df = pd.DataFrame(data=hourly_data)
print(f"✅ Datos obtenidos para: {response.Latitude()}°N {response.Longitude()}°E")
"""

'\nurl = "https://api.open-meteo.com/v1/forecast"\nparams = {\n    "latitude": 7.9,\n    "longitude": -72.5,\n    "hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "rain"],\n    "timezone": "America/Bogota",\n    #"past_days": 7, # Aumentamos a 7 días para mejor análisis\n    #"forecast_days": 0\n    "start_date": "2026-05-06", \n    "end_date": "2026-05-13"\n}\n\nresponses = openmeteo.weather_api(url, params=params)\nresponse = responses[0]\n\n# Procesar variables horarias\nhourly = response.Hourly()\nhourly_data = {\n    "datetime": pd.date_range(\n        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),\n        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),\n        freq=pd.Timedelta(seconds=hourly.Interval()),\n        inclusive="left",\n    ),\n    "temperature": hourly.Variables(0).ValuesAsNumpy(),\n    "humidity": hourly.Variables(1).ValuesAsNumpy(),\n    "precipitation": hourly.Variables(2).ValuesAsNumpy(),\n    "rain": hourly.Variables(3

In [53]:
datos="../data/clima_cucuta_mayo_2026.csv"

In [54]:
"""
df.to_csv(datos, index=False)
print("✅ Datos guardados localmente.")
"""

'\ndf.to_csv(datos, index=False)\nprint("✅ Datos guardados localmente.")\n'

In [55]:
df = pd.read_csv(datos)
df['datetime'] = pd.to_datetime(df['datetime'])

In [56]:
# Crear copia y ajustar zona horaria
df_cucuta = df.copy()
df_cucuta["datetime"] = df_cucuta["datetime"].dt.tz_convert("America/Bogota")

# Extraer características temporales
df_cucuta["hour"] = df_cucuta["datetime"].dt.hour
df_cucuta["date"] = df_cucuta["datetime"].dt.date # Solo fecha para agrupaciones

# Verificar nulos
print("Nulos encontrados:\n", df_cucuta.isnull().sum())

# Mostrar los primeros datos procesados
df_cucuta.head()

Nulos encontrados:
 datetime         0
temperature      0
humidity         0
precipitation    0
rain             0
hour             0
date             0
dtype: int64


,datetime,temperature,humidity,precipitation,rain,hour,date
0,2026-05-06 00:00:00-05:00,24.90,80.43464,0.0,0.0,0,2026-05-06
1,2026-05-06 01:00:00-05:00,24.50,87.30226,0.0,0.0,1,2026-05-06
2,2026-05-06 02:00:00-05:00,25.50,78.80969,0.0,0.0,2,2026-05-06
3,2026-05-06 03:00:00-05:00,25.05,77.54565,0.0,0.0,3,2026-05-06
4,2026-05-06 04:00:00-05:00,24.60,76.52748,0.1,0.1,4,2026-05-06


## 📊 1. Análisis Descriptivo (KPIs)
Antes de visualizar, calculamos los indicadores clave del clima en el periodo seleccionado.

In [57]:
# Cálculos de métricas
temp_avg = df_cucuta["temperature"].mean()
temp_max = df_cucuta["temperature"].max()
rain_total = df_cucuta["rain"].sum()
hum_avg = df_cucuta["humidity"].mean()

print(f"🌡️ Temperatura Promedio: {temp_avg:.2f}°C")
print(f"🔥 Temperatura Máxima: {temp_max:.2f}°C")
print(f"🌧️ Precipitación Total: {rain_total:.2f} mm")
print(f"💧 Humedad Promedio: {hum_avg:.2f}%")

🌡️ Temperatura Promedio: 28.80°C
🔥 Temperatura Máxima: 35.05°C
🌧️ Precipitación Total: 0.10 mm
💧 Humedad Promedio: 57.74%


In [58]:
# ¿A qué hora llueve más en promedio?
lluvia_por_hora = df_cucuta.groupby("hour")["rain"].mean().sort_values(ascending=False)

print("⏰ Top 5 horas con más probabilidad de lluvia (promedio):")
print(lluvia_por_hora.head(5))

⏰ Top 5 horas con más probabilidad de lluvia (promedio):
hour
4    0.0125
0    0.0000
2    0.0000
1    0.0000
3    0.0000
Name: rain, dtype: float64


In [59]:
# Usamos Plotly para que la matriz también sea interactiva
corr_matrix = df_cucuta[["temperature", "humidity", "precipitation", "rain"]].corr()

fig_corr = px.imshow(
    corr_matrix, 
    text_auto=".2f", 
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title="Matriz de Correlación Interactiva"
)
fig_corr.show()

## 🔍 Análisis de Correlaciones

### 1. Temperatura vs. Humedad (**-0.94**)
Se observa una **correlación negativa casi perfecta**. Esto indica una relación inversamente proporcional extrema:
*   **Hallazgo:** Cuando la temperatura sube, la humedad cae drásticamente.
*   **Interpretación:** En Cúcuta, el calor es predominantemente "seco" durante las horas pico. Este fenómeno es típico de climas donde el sol calienta rápidamente el aire, reduciendo su capacidad de retener humedad relativa.

### 2. Precipitación vs. Lluvia (**0.57**)
Existe una **correlación positiva moderada**. 
*   **Hallazgo:** Aunque están relacionadas, no son lo mismo. La precipitación incluye otros fenómenos (como humedad condensada o lloviznas leves), mientras que la variable `rain` se enfoca en lluvia líquida medible.
*   **Interpretación:** Un valor de 0.57 sugiere que hubo momentos de alta humedad o nubosidad (precipitación) que no necesariamente terminaron en una lluvia fuerte.

### 3. La Independencia de la Lluvia
Las correlaciones de la lluvia con la temperatura (**-0.09**) y la humedad (**0.10**) son **casi nulas**.
*   **Hallazgo:** La lluvia en este periodo no dependió de si el día estaba frío o caluroso.
*   **Interpretación:** Esto sugiere que las lluvias en Cúcuta durante estos días fueron **eventos aislados o chubascos repentinos**, que no alteraron la tendencia térmica general de la ciudad. No se puede predecir la lluvia basándose únicamente en la temperatura del momento.

In [60]:
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.08, 
    subplot_titles=("Evolución de la Temperatura (°C)", "Evolución de la Precipitación (mm)")
)

fig.add_trace(
    go.Scatter(
        x=df_cucuta['datetime'], 
        y=df_cucuta['temperature'], 
        name="Temperatura",
        mode='lines',
        line=dict(color="#FF5733", width=2)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_cucuta['datetime'], 
        y=df_cucuta['rain'], 
        name="Lluvia",
        mode='lines',
        line=dict(color="#337BFF", width=2),
        fill='tozeroy' 
    ),
    row=2, col=1
)

fig.update_layout(
    height=700,
    title_text="<b>Análisis Climático de Cúcuta</b>",
    showlegend=False,
    template="plotly_white",
    hovermode="x unified"
)

fig.update_xaxes(rangeslider_visible=False) 

# Nombres de ejes
fig.update_yaxes(title_text="Grados °C", row=1, col=1)
fig.update_yaxes(title_text="Milímetros mm", row=2, col=1)

fig.show()

## 📈 2. Análisis de Series Temporales: Temperatura y Lluvia

Al observar el comportamiento cronológico de los últimos 7 días en Cúcuta, podemos extraer conclusiones valiosas sobre su dinámica climática:

### A. Ciclo Circadiano Perfecto
La gráfica de temperatura muestra una **oscilación sinusoidal casi perfecta**. 
*   **Picos de Calor:** Cada día, la temperatura escala rápidamente desde los 24°C hasta alcanzar picos de **34°C - 35°C** entre las 2:00 PM y las 4:00 PM.
*   **Recuperación Térmica:** Durante las madrugadas, la ciudad logra enfriarse hasta los 23°C - 24°C, lo que representa una **amplitud térmica de aproximadamente 11°C**. Esto es típico de climas con cielos despejados donde el calor se escapa rápidamente al espacio durante la noche (irradiación).

### B. Ausencia de Precipitaciones (Estrés Hídrico)
La gráfica de precipitación revela un dato crítico:
*   **Evento Único:** Solo se registró un evento de lluvia muy leve (0.1 mm) al inicio del periodo (6 de mayo). 
*   **Sequedad Extrema:** El resto de la semana la precipitación fue de **0.0 mm**. 
*   **Impacto:** Esta falta de lluvia explica por qué las curvas de temperatura son tan consistentes y "limpias". Sin nubes ni lluvia que refresquen el ambiente, el sol calienta la superficie de manera ininterrumpida día tras día.


In [61]:
fig_hum = px.line(
    df_cucuta, 
    x='datetime', 
    y='humidity', 
    title='Variación de la Humedad Relativa (%)',
    color_discrete_sequence=['#2ECC71']
)
fig_hum.update_layout(template="plotly_dark")
fig_hum.show()

## 💧 3. Variación de la Humedad Relativa: El Espejo Térmico

La humedad relativa en Cúcuta muestra un comportamiento cíclico extremo, fundamental para entender la sensación térmica de la ciudad:

### A. Correlación Inversa con la Temperatura
Al comparar esta gráfica con la de temperatura, observamos un fenómeno de **"espejo invertido"**:
*   **Madrugadas Húmedas:** Entre las 4:00 AM y las 7:00 AM, la humedad alcanza sus picos máximos (entre **80% y 92%**). Esto ocurre porque el aire frío tiene menos capacidad de retener vapor de agua, acercándose al punto de rocío.
*   **Tardes Secas:** Entre las 2:00 PM y las 5:00 PM, la humedad cae a niveles críticos de **30% a 40%**. El calor intenso "diluye" la humedad relativa, generando un ambiente de calor seco.

### B. El Pico del 9 de Mayo
Es notable que el **9 de mayo** se registró el punto más alto de humedad del periodo (superando el 90%). 
*   **Interpretación:** Aunque no se registró lluvia fuerte ese día, este pico sugiere una alta nubosidad o un aumento en el transporte de humedad desde la cuenca del Catatumbo o Venezuela, lo que probablemente generó una sensación de "bochorno" o pesadez climática mayor que en los días anteriores.

### C. Estabilidad del "Confort" Climático
A pesar de los picos nocturnos, la ciudad pasa más del 50% del día por debajo del 60% de humedad. 
*   **Insight:** Esto confirma que Cúcuta, a pesar de su calor, no tiene un clima tropical húmedo constante (como el de una selva), sino que experimenta una oscilación que permite la evaporación rápida del sudor durante las horas de mayor sol.


# 🏁 Conclusiones Finales: El Perfil Climático de Cúcuta

Tras completar el Análisis Exploratorio de Datos (EDA) y la construcción del Dashboard interactivo, se presentan las conclusiones definitivas sobre el comportamiento atmosférico de la ciudad en el periodo analizado:

### 1. Dominancia del Ciclo Solar y Estabilidad Térmica
Cúcuta presenta una **estabilidad climática excepcional**. La temperatura sigue un patrón predictivo casi perfecto, con picos de **34°C - 35°C** y valles de **24°C**. La ausencia de lluvias significativas durante la semana permitió que el ciclo de radiación solar dictara el ritmo del clima sin interferencias, generando una amplitud térmica diaria de 11°C.

### 2. La Correlación Inversa como Motor del Clima
El hallazgo estadístico más potente fue la correlación de **-0.94** entre temperatura y humedad. 
*   Este número no es solo una estadística; describe la **sensación térmica** de la ciudad: un calor intenso que "seca" el ambiente cada tarde. 
*   A diferencia de otras ciudades tropicales donde el calor viene acompañado de alta humedad (bochorno constante), Cúcuta experimenta un alivio en la humedad relativa precisamente cuando el calor es más fuerte, facilitando la evaporación.

### 3. Diagnóstico de Estrés Hídrico
El análisis de precipitación reveló un estado de **sequedad absoluta** (0.0 mm de lluvia en el 98% del tiempo analizado). Para una ciudad con temperaturas tan altas, la falta de lluvia prolongada sugiere un aumento en la demanda de energía (climatización) y un posible estrés en la vegetación urbana. El pico de humedad del **9 de mayo** sin lluvia asociada indica un aumento de nubosidad que no llegó a descargar, un fenómeno común en valles rodeados de montañas.

### 4. Valor Técnico del Proyecto
Desde la perspectiva de Ciencia de Datos, este proyecto demuestra la capacidad de:
*   **Automatizar la obtención de información** mediante el consumo de APIs profesionales (Open-Meteo).
*   **Garantizar la integridad de los datos** mediante el manejo de zonas horarias y limpieza de nulos.
*   **Comunicar insights complejos** a través de visualizaciones interactivas en Plotly, permitiendo que cualquier usuario entienda la relación entre variables sin necesidad de ver el código fuente.

---
**Análisis finalizado con éxito.** Este dashboard queda como una herramienta funcional para el monitoreo climático y la toma de decisiones basada en datos.